# Volume 11, Chapter 2: The fine structure of a Brownian path (Python companion)

Exercises first, solutions at the end. Lévy's modulus of continuity seen on long paths, the joint law of the maximum and the position, the probability of no zero in an interval, the zero set and the record set measured by **box counting**, and a computer science application: **estimating fractal dimensions from data**, the box-counting algorithm used in image analysis, and why it converges so slowly. Requires `numpy`.

In [ ]:
import math
import numpy as np
rng = np.random.default_rng(1102)

def bm(n, m=1):
    # m Brownian paths at the 2**n + 1 dyadic points of [0, 1]
    N = 2**n
    steps = rng.standard_normal((m, N)) / math.sqrt(N)
    return np.concatenate([np.zeros((m, 1)), np.cumsum(steps, axis=1)], axis=1)

## Exercises

**P2.1 (★) Lévy's modulus.** For one path with $2^{20}$ steps, compute $R(h)=\max_t|B_{t+h}-B_t|/\sqrt{2h\log(1/h)}$ for $h=2^{-k}$, $k=2,\dots,18$. Theorem 2.4 says $\limsup_{h\to0}R(h)=1$. Is the ratio close to $1$ already? Repeat for $20$ paths. Two effects compete at finite $h$: the discrete grid misses the largest increments (pushing the ratio down), and the next term of the asymptotics pushes it up. Which wins?

**P2.2 (★) The maximum and the endpoint.** Simulate $(M_1,B_1)$ for $20000$ paths of $2^{12}$ steps. Check Theorem 2.5 at $(a,b)=(1,0)$: $P(M_1\ge1,B_1\le0)=P(B_1\ge2)$. Check Exercise 2.3: given $|B_1|$ small, the maximum of the path exceeds $a$ with probability about $e^{-2a^2}$.

**P2.3 (★★) No zero in an interval.** Estimate $P(\text{no zero in }]a,1[)$ for $a=0.1,0.3,0.5,0.9$ and compare with $\frac2\pi\arcsin\sqrt a$ (Exercise 2.4). Which way is the bias of the discrete simulation, and why?

**P2.4 (★★) Box counting the zero set and the record set.** For a path with $2^{22}$ steps, count the number $N(\varepsilon)$ of intervals of length $\varepsilon=2^{-k}$ that contain a sign change (zero set) or a new running maximum (record set), for $k=4,\dots,18$. Fit $\log N\approx d\log(1/\varepsilon)$. Compare with Theorem 2.9 (and, for the zero set, with Exercise 2.6 and the completion in Chapter 4).

**P2.5 (★★★) CS application: how good is box counting?** Box counting is the standard algorithm for the fractal dimension of an image or a time series. (a) Apply it to the middle-thirds Cantor set (exact dimension $\log2/\log3\approx0.631$), generated to depth $14$, with boxes of side $3^{-k}$ and with boxes of side $2^{-k}$. Why do the two answers differ at moderate scales? (b) Apply it to the graph of a Brownian path (dimension $\frac32$) using square boxes. (c) For the zero set, the expected count is $\mathbb E N(\varepsilon)\approx c\,\varepsilon^{-1/2}$; estimate the fluctuations of $N(\varepsilon)/\varepsilon^{-1/2}$ across $50$ paths. Is the limit random? (It is: the local time at $0$.)

## Solutions

### P2.1
**How to find it.** Differences at lag $2^{n-k}$ on the grid give all increments over intervals of length $2^{-k}$ whose endpoints are grid points. The continuous path has larger increments (between grid points), so the discrete maximum underestimates. But at moderate $h$ there are only $1/h$ roughly independent increments, and the maximum of $N$ Gaussians is $\sqrt{2\log N}$ plus fluctuations of order $1/\sqrt{\log N}$, which are large when $\log N$ is small: at $h=2^{-2}$ the ratio is meaningless, and it settles near $1$, slightly above, only for $h\le2^{-8}$.

In [ ]:
n = 20
B = bm(n)[0]
for k in range(2, 19, 2):
    lag = 2**(n - k); h = 2.0**-k
    D = np.abs(B[lag:] - B[:-lag]).max()
    print(f"h = 2^-{k:2d}: ratio = {D / math.sqrt(2*h*math.log(1/h)):.3f}")
ratios = []
for _ in range(20):
    B = bm(18)[0]; lag = 2**(18 - 12); h = 2.0**-12
    ratios.append(np.abs(B[lag:] - B[:-lag]).max() / math.sqrt(2*h*math.log(1/h)))
print("20 paths at h = 2^-12: mean ratio", round(float(np.mean(ratios)), 3), " range", round(min(ratios), 3), round(max(ratios), 3))

### P2.2
**How to find it.** Keep the maximum along each path and the endpoint. For the conditional statement, select the paths with $|B_1|<0.05$ (an approximation of conditioning on $B_1=0$).

In [ ]:
B = bm(12, 20000)
M = B.max(axis=1); E = B[:, -1]
from math import erf, sqrt
Phi = lambda x: 0.5 * (1 + erf(x / sqrt(2)))
print("P(M>=1, B<=0) =", np.mean((M >= 1) & (E <= 0)), "  P(B>=2) =", round(1 - Phi(2), 4))
sel = np.abs(E) < 0.05
for a in (0.5, 1.0):
    print(f"P(M>={a} | B_1 ~ 0) = {np.mean(M[sel] >= a):.3f}   exp(-2a^2) = {math.exp(-2*a*a):.3f}")

The simulated maximum is a maximum over $4097$ points only, so it is a little too small. The conditional estimate uses only the paths with $|B_1|<0.05$, about $800$ of them, so its sampling error is about $0.012$: the agreement is within the noise.

### P2.3
**How to find it.** A zero of the continuous path between two grid points of the same sign is missed by the simulation, so the simulated probability of *no* zero is too large.

In [ ]:
B = bm(12, 20000); N = 2**12
for a in (0.1, 0.3, 0.5, 0.9):
    seg = B[:, int(a * N):]
    nozero = np.all(seg > 0, axis=1) | np.all(seg < 0, axis=1)
    print(f"a = {a}: simulated {nozero.mean():.3f}   (2/pi) arcsin sqrt(a) = {2/math.pi*math.asin(math.sqrt(a)):.3f}")

### P2.4
**How to find it.** A box $[j\varepsilon,(j+1)\varepsilon]$ meets the zero set if the path changes sign (or vanishes) inside it, up to the grid resolution; it meets the record set if the running maximum increases inside it. Both counts should grow like $\varepsilon^{-1/2}$.

In [ ]:
n = 22; N = 2**n
B = bm(n)[0]
sign_change = np.concatenate([[True], (B[1:] * B[:-1] <= 0)])
running = np.maximum.accumulate(B)
record = np.concatenate([[True], running[1:] > running[:-1]])
ks = range(4, 19); Nz = []; Nr = []
for k in ks:
    box = 2**(n - k)
    Nz.append(sign_change[:N].reshape(2**k, box).any(axis=1).sum())
    Nr.append(record[:N].reshape(2**k, box).any(axis=1).sum())
x = np.array(list(ks)) * math.log(2)
print("zero set dimension estimate:  ", round(np.polyfit(x, np.log(Nz), 1)[0], 3))
print("record set dimension estimate:", round(np.polyfit(x, np.log(Nr), 1)[0], 3))

### P2.5
**How to find it.** Box counting measures the growth exponent of $N(\varepsilon)$, but at moderate scales the boxes may be badly aligned with the set (part a), and the prefactor of $\varepsilon^{-d}$ may itself be random (part c), which adds noise to every fit. For the graph, a box column above an interval of length $\varepsilon$ needs about (oscillation)$/\varepsilon\approx\varepsilon^{-1/2}$ boxes, so $N(\varepsilon)\approx\varepsilon^{-3/2}$.

In [ ]:
# (a) Cantor set: left endpoints of the 2^14 intervals of level 14
depth = 14
pts = np.zeros(1)
for d in range(1, depth + 1):
    pts = np.concatenate([pts, pts + 2 * 3.0**-d])
for base in (3, 2):
    ks = range(2, 12) if base == 3 else range(3, 19)
    cnt = [len(np.unique(np.floor(pts * base**k + 1e-9))) for k in ks]
    est = np.polyfit(np.array(list(ks)) * math.log(base), np.log(cnt), 1)[0]
    print(f"Cantor set, boxes of side {base}^-k: dimension estimate {est:.3f} (exact {math.log(2)/math.log(3):.3f})")
# (b) graph of a Brownian path
n = 16; N = 2**n; B = bm(n)[0]
cnts = []; ks = range(3, 12)
for k in ks:
    box = 2**(n - k); eps = 2.0**-k
    cols = B[:N].reshape(2**k, box)
    hi = np.maximum(cols.max(axis=1), B[box::box]); lo = np.minimum(cols.min(axis=1), B[box::box])
    cnts.append(np.sum(np.floor(hi / eps) - np.floor(lo / eps) + 1))
print("graph dimension estimate:", round(np.polyfit(np.array(list(ks)) * math.log(2), np.log(cnts), 1)[0], 3), "(exact 1.5)")
# (c) the random prefactor of the zero count
vals = []
for _ in range(50):
    B = bm(16)[0]
    sc = (B[1:] * B[:-1] <= 0)
    k = 12; box = 2**(16 - k)
    vals.append(sc.reshape(2**k, box).any(axis=1).sum() / 2**(k / 2))
print("N(eps) eps^(1/2) at eps = 2^-12 over 50 paths: mean", round(np.mean(vals), 3), " std", round(np.std(vals), 3))

The base-$3$ boxes are aligned with the Cantor set and give the exact answer; base-$2$ boxes cut it at the wrong places and the estimate oscillates with the scale (the Cantor set is *not* self-similar under halving). The rescaled zero count fluctuates from path to path by a random factor of order one: it converges, as $\varepsilon\to0$, to a multiple of the local time of the path at zero (Chapter 4), not to a constant. Dimension estimates from a single trace are therefore only as good as the range of scales available.